### 此Jupyter展示SeisMoLLM在震相拾取、反方位角估算、震中距估算、震级估算与初动极性分类这五种任务中的模型应用示例，流程如下：
1. 实例化对应任务的SeisMoLLM模型并加载预训练权重
2. 读取所需数据样本，并将通道顺序调整为[z, n, e]
3. 标准化数据样本并转换为[batch=1, channel, length]形状
4. 将数据样本输入模型进行推理
5. 对模型的输出预测进行后处理或可视化得到最终结果

In [ ]:
import os
import math
import h5py
import numpy as np
import pandas as pd
import torch
from typing import Dict, Union
from models import create_model, load_checkpoint
from utils import vis_phase_picking

import warnings
warnings.filterwarnings("ignore")

# 预先统一指定所用硬件, 可选 "cpu" 或 "cuda"(即GPU)
device = "cpu" # "cpu" | "cuda"

##### 定义模型加载函数

In [ ]:
def load_model(
    task: str,  # "dpk" | "baz" | "dis" | "emg" | "pmp"
    ckpt_path: str,
    in_channels: int = 3,
    in_samples: int = 8192, # "6000"
    conv_channels = [16, 48, 96]
):
    '''
    task: 任务名缩写str, 可选 dpk(震相拾取), baz(反方位角), dis(震中距), emg(震级), pmp(初动极性)
    ckpt_path: 模型权重文件路径str
    in_channels: 输入数据通道数, 默认为3无需更改
    in_samples: 输入数据的序列长度, 使用STEAD训练的模型时=6000, DiTing模型时=8192
    conv_channels: 多尺度卷积Embedder的通道list, 传入create_model()以避免重复实例化时的报错, 无需更改
    '''
    # Model init
    model = create_model(
        model_name=f"SeisMoLLM_{task}", in_channels=in_channels, 
        in_samples=in_samples, conv_channels = conv_channels
    )

    # Load parameters
    ckpt = load_checkpoint(ckpt_path, device=device)
    model_state_dict = ckpt["model_dict"] if "model_dict" in ckpt else ckpt
    model.load_state_dict(model_state_dict)
    model.eval().to(device)  # 别忘记.eval()

    return model

##### 定义数据标准化函数

In [ ]:
def normalize(data):
    """
        标准化波形, 减均值除以标准差
        input:
            data : 2D array like, [channel, length]
        ouput:
            data : 2D array 
    """
    data -= np.mean(data, axis=1, keepdims=True)
    std_data = np.std(data, axis=1, keepdims=True)
    std_data[std_data == 0] = 1
    data /= std_data
    
    return data

##### 定义样本数据与对应label加载函数

In [ ]:
# 任务缩写到列名的映射
TASK_MAP = {"dpk": None, "baz": "baz", "dis": "dis", "emg": "evmag", "pmp": "p_motion"}
def choose_columns(task: str) -> list[str]:
    '''   根据目标任务来灵活选择meta csv中需要读取的label列, 默认包括phase picking   '''
    return ["part", "key", "p_pick", "s_pick", TASK_MAP[task]]


def load_diting(
    dataset_path: str,
    meta_file: str,
    trace_name: Union[str, int],
    task: str
):
    '''
    参数:
        dataset_path: 数据集文件路径str
        meta_file: meta csv文件名str(如"DiTing330km_light.csv")
        trace_name: 支持两种模式, 根据样本名str(如"000014.0100")以精准检索, 或传入样本在meta csv中的int序号
        task: 目标任务名缩写str, 据此灵活选择在meta_file读取的列
    返回:
        data: 样本数据tensor
        labels: 以任务名缩写为键, 记录对应label值的字典
    注意, 当前示例灵活读取task所需的label, 并对样本缺失所需label的情况告警
    DiTing数据集通道顺序本就是ZNE, 无需调整
    '''  

    # 获取需要读取的列名与各列的转换操作集合
    cols = choose_columns(task)
    # 根据任务灵活读取metadata csv, 获取全部样本ID与label
    meta_df = pd.read_csv(os.path.join(dataset_path, meta_file), usecols=cols,
        dtype={c: t for c, t in {"part":int, "key":str, "p_pick":int, "s_pick":int,
                                 "baz":float, "dis":float, "evmag":float, "p_motion":str}.items() if c in cols})

    # 根据样本名str或序号检索
    if isinstance(trace_name, int):
        target_event = meta_df.iloc[trace_name]
        part = target_event["part"]
        key = target_event["key"]
        key_correct = key.split(".")
        key = key_correct[0].rjust(6, "0") + "." + key_correct[1].ljust(4, "0")
    elif isinstance(trace_name, str):
        target_event = meta_df[meta_df["key"]==trace_name]
        key = trace_name
    else:
        raise ValueError(f"Supported trace_name type: int or str")
    
    # 将所有label以任务名缩写为键保存至label字典, 并将初动极性的str label转换为0-1二值编码, 0代表向上, 1代表向下
    labels = {}
    if target_event.isna().any(): 
        raise ValueError(f"{task} label of {key} sample is not avaible, please change the sample or exclude the task")
    else:
        labels["p_arr"], labels["s_arr"], labels[task] = target_event["p_pick"], target_event["s_pick"], target_event[TASK_MAP[task]]
        if task == "pmp":
            labels["pmp"] = {"u": 0, "c": 0, "r": 1, "d": 1}[labels["pmp"].lower()]

    # 在对应数据切片中检索样本
    path = os.path.join(dataset_path, f"DiTing330km_part_{part}.hdf5")
    with h5py.File(path, "r", libver='latest', swmr=True) as f:
        data = f.get("earthquake/" + str(key))
        data = np.array(data).astype(np.float32).T  # 注意不要遗漏.T, 转置样本维度为[channel, length]
        data = data[:, :8192]  # 注意裁剪样本长度以匹配模型输入长度

    # 标准化与类型、形状转化
    data = torch.from_numpy(normalize(data)).reshape(1, 3, -1).to(device)
    print(f"Using sample {key}, shape: {data.shape}")  # shape应为[1, 3, 8192]

    return data, labels

In [ ]:
def load_stead(
    dataset_path: str,
    meta_file: str,
    trace_name: Union[str, int],
    task: str
):
    '''
    参数:
        dataset_path: 数据集文件路径str
        meta_file: meta csv文件名str(如"merge.csv")
        trace_name: 支持两种模式, 根据样本名str(如"000014.0100")以精准检索, 或传入样本在meta csv中的int序号
        task: 目标任务名缩写str
    返回:
        data: 样本数据tensor
        labels: 以任务名缩写为键, 记录对应label值的字典
    注意, 对于STEAD数据集的震级估算, 我们只使用了ML震级类型数据; STEAD原通道顺序为ENZ, 勿忘调整顺序!
    '''  

    # 读取metadata csv, 获取全部样本ID与label
    meta_df = pd.read_csv(os.path.join(dataset_path, meta_file), 
                dtype={"trace_name": str, "trace_category": str, "source_id": str, "p_arrival_sample": float, "s_arrival_sample": float, 
                       "back_azimuth_deg": float, "source_distance_km": float, "source_magnitude": float, "source_magnitude_type": str}, index_col=0)
    # 剔除噪声样本, 只保留地震样本
    meta_df = meta_df[meta_df["trace_category"] == "earthquake_local"]
    # 震级估算只使用了ML类型数据
    if task == "emg":
        meta_df = meta_df[meta_df["source_magnitude_type"] == "ml"]
    
    # 根据样本名str或序号检索
    if isinstance(trace_name, int):
        target_event = meta_df.iloc[trace_name]
        key = target_event["trace_name"]
    elif isinstance(trace_name, str):
        key = trace_name
    else:
        raise ValueError(f"Supported trace_name type: int or str")
    
    # 将所有label以任务名缩写为键保存至label字典
    labels = {}
    labels["p_arr"], labels["s_arr"] = target_event["p_arrival_sample"], target_event["s_arrival_sample"]
    labels["baz"], labels["dis"], labels["emg"] = target_event["back_azimuth_deg"] % 360, target_event["source_distance_km"], target_event["source_magnitude"]

    # 在对应数据切片中检索样本
    path = os.path.join(dataset_path, f"part_0.hdf5")
    with h5py.File(path, "r", libver='latest', swmr=True) as f:
        data = f.get("data/" + str(key))
        data = np.array(data).astype(np.float32).T  # 注意不要遗漏.T, 转置样本维度为[channel, length]
        data = data[:, :6000]  # 注意裁剪样本长度以匹配模型输入长度

    # 标准化与类型、形状转化
    data = torch.from_numpy(normalize(data)).reshape(1, 3, -1).to(device)
    # 将[B, C, L]中的channel维度顺序由ENZ调整为ZNE
    data[:, [0, 2], :] = data[:, [2, 0], :]
    print(f"Using sample {key}, shape: {data.shape}")  # shape应为[1, 3, 6000]

    return data, labels

##### 定义处理模型输出预测的后处理函数

In [ ]:
def detect_single_sequence(prob_seq, threshold=0.3, min_sec=1.0, dt=0.01, padding=0):
    """
    使用峰值检测逻辑从单一概率序列中拾取震相。连续高于阈值的采样点按“最小时间间隔”合并。
    参数：
        prob_seq(1维 np.ndarray): 概率序列
        threshold(float): 最低概率阈值(detect_peaks 中的 mph)
        min_sec(float): 峰与峰之间的最小时间间隔, 单位秒
        dt(float): 采样间隔，单位秒
    返回：
        picks_sample(np.ndarray): 检测到的峰值在序列中的索引
        picks_prob(np.ndarray): 对应峰值处的概率值
    """
    if padding > 0:
        prob_seq = np.pad(prob_seq, (padding, padding), mode='constant', constant_values=0)

    prob_seq = np.atleast_1d(prob_seq).astype('float64')
    if prob_seq.size < 3:
        return np.array([], dtype=int), np.array([], dtype=float)

    # convert min_sec to samples
    mpd = max(1, int(min_sec / dt))

    # compute derivative
    dx = prob_seq[1:] - prob_seq[:-1]

    # detect rising edges (local maxima)
    ind = np.where((np.hstack((dx, 0)) <= 0) & (np.hstack((0, dx)) > 0))[0]

    # remove first/last point
    if ind.size and ind[0] == 0:
        ind = ind[1:]
    if ind.size and ind[-1] == prob_seq.size - 1:
        ind = ind[:-1]

    # filter by threshold
    if ind.size:
        ind = ind[prob_seq[ind] >= threshold]

    # apply minimum peak distance (mpd)
    if ind.size and mpd > 1:
        # sort by probability descending
        ind = ind[np.argsort(prob_seq[ind])][::-1]
        idel = np.zeros(ind.size, dtype=bool)
        for i in range(ind.size):
            if not idel[i]:
                idel = idel | ((ind >= ind[i] - mpd) & (ind <= ind[i] + mpd))
                idel[i] = 0  # keep current
        ind = np.sort(ind[~idel])

    picks_sample = np.array([int(i) for i in ind])
    picks_prob = np.array([float(prob_seq[i]) for i in ind])

    return picks_sample, picks_prob

def post_process(task: str, train_data:str, data, prediction, labels):
    '''
    后处理函数, 只打印出模型预测与真实结果并在phase picking时保存可视化图, 无返回值
    参数:
        task: 任务名缩写str
        train_data: 模型所用训练集名str
        data: 样本数据tensor
        prediction: 模型输出的预测
        labels: 从meta csv读取的真实label字典
    对于不同任务的prediction:
        dpk: 与输入data形状一致的概率序列tensor, 第一个channel无意义, 第2、3 channel分别对应P波与S波到时概率
        baz: 预测的反方位角的(cos, sin)元组
        dis & emg: 震中距与震级的预测值
        pmp: 对应(极性为上概率, 极性为下概率)的独热编码
    '''
    if task == "dpk":
        if train_data == "stead":
            sampling_rate, waveforms_labels = 100, ["E", "N", "Z"]
        else:
            sampling_rate, waveforms_labels = 50, ["Z", "N", "E"]

        # 拾取预测概率最大处, 阈值0.3, 最小间隔0.5s
        pred = prediction.detach().cpu().numpy().reshape(3, -1)[1:]  # SeisMoLLM不进行检测任务, 舍去第一个无意义channel
        ppicks_sample, ppicks_prob = detect_single_sequence(pred[0], threshold=0.3, min_sec=0.5, dt=1/sampling_rate)
        spicks_sample, spicks_prob = detect_single_sequence(pred[1], threshold=0.3, min_sec=0.5, dt=1/sampling_rate)
        print(f"Model predicts P wave onset is {ppicks_sample} with probability of {ppicks_prob}, true onset is {labels['p_arr']}")
        print(f"Model predicts S wave onset is {spicks_sample} with probability of {spicks_prob}, true onset is {labels['s_arr']}")

        # 可视化, 可以修改save_dir与save_name来更改保存的路径与文件名
        vis_phase_picking(data.detach().cpu().numpy(), waveforms_labels, pred, [labels["p_arr"], labels["s_arr"]], 
                          ["P", "S"], ["$\hat{P}$", "$\hat{S}$"], None, save_name=f"phase_picking", save_dir="./")
    
    elif task == "baz":
        # 将模型输出转化为反方位角的预测浮点数值(角度)
        pred = round((torch.atan2(prediction[1], prediction[0])).item() * 180 / math.pi, 1)
        # 计算正确error值
        error = np.abs(labels[task] - pred)
        error = np.where(error > 360, error - 360, error)
        error = np.where(error > 180, 360 - error, error)
        print(f"Model predicts back-azimuth is {pred}, truth is {labels[task]}, and error is {round(error, 2)}")

    elif task == "dis":
        print(f"Model predicts epicentral distance is {round(prediction.item(), 3)} km, and truth is {labels[task]} km")

    elif task == "emg":
        print(f"Model predicts magnitude is {round(prediction.item(), 2)}, and truth is {labels[task]}")

    else:
        pred, prob = torch.argmax(prediction, dim=-1).item(), prediction.max().item()
        pred, label = {"0": "up", "1": "down"}[str(pred)], {"0": "up", "1": "down"}[str(labels[task])]
        print(f"Model predicts P-wave first-motion polarity is {pred} with probability of {prob}, and truth is {label}")

##### 主函数：实例化模型、读取实际数据、将样本输入模型进行推理

In [ ]:
task = "baz"
train_data = "stead" # "ditinglight"
sample_len = 6000 if train_data == "stead" else 8192

# 加载模型与权重
model = load_model(task, ckpt_path=f"/home/bingxing2/ailab/wangxinghao/project/seist/results/seismollm/{task}_{train_data}.pth", in_samples=sample_len, conv_channels = [16, 48, 96])
# 出现Some weights of GPT2Model were not initialized ... wte.weight: found shape torch.Size([50257, 768]) in the checkpoint and torch.Size([0, 768]) ...是正常的

In [ ]:
for i in (0, 100, 500, 1000, 5000, 10000):
    # 读取样本数据与任务所需label (注意STEAD数据集不支持pmp任务), 并对数据进行标准化与转换
    data_tensor, label_dict = load_stead(dataset_path="/home/bingxing2/ailab/group/ai4earth/wangxinghao/STEAD/stead_fast_parts", 
                                        meta_file="eq.csv", trace_name=i, task=task)
    # 将样本输入模型进行推理
    prediction = model(data_tensor)

    post_process(task, train_data, data_tensor.squeeze(), prediction, label_dict)

In [ ]:
task = "baz"
train_data = "ditinglight"
sample_len = 6000 if train_data == "stead" else 8192

# 加载模型与权重
model = load_model(task, ckpt_path=f"./results/seismollm/{task}_{train_data}.pth", in_samples=sample_len, conv_channels = [16, 48, 96])
# 出现Some weights of GPT2Model were not initialized ... wte.weight: found shape torch.Size([50257, 768]) in the checkpoint and torch.Size([0, 768]) ...是正常的

In [ ]:
# 批量运行示例
for i in (0, 100, 500, 1000, 5000, 10000):
    # 读取样本数据与任务所需label (注意STEAD数据集不支持pmp任务), 并对数据进行标准化与转换
    data_tensor, label_dict = load_diting(dataset_path="/home/bingxing2/ailab/group/ai4earth/wangxinghao/DiTing330km", 
                                        meta_file="DiTing330km_light.csv", trace_name=i, task=task)
    # 将样本输入模型进行推理
    prediction = model(data_tensor)

    post_process(task, train_data, data_tensor.squeeze(), prediction, label_dict)